In [ ]:
import os
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import re
import datetime
import statsmodels.api as sm
import requests
import apimoex

## ISIN list in Y data

In [ ]:
Y_funds_df = pd.read_excel(r'../../Data/Y/Y.xlsx')

In [ ]:
ISIN_list = Y_funds_df['ISIN Фонда'].dropna().unique().tolist()
ISIN_list = [isin for isin in ISIN_list if not str(isin).startswith("US")]

In [ ]:
def stocks_quotes_download(isin, start = '2024-01-01'):
    with requests.Session() as session:
            try:
                data = apimoex.get_market_history(session=session,security=isin,  start = start, market="shares",engine="stock")
                df_data = pd.DataFrame(data)
                # df_data = df_data[df_data['BOARDID'] == 'TQBR']
                df_data.insert(0,'ISIN',isin)
            except:None
    return (df_data)

In [ ]:
all_data = []
for isin in ISIN_list:
    try:
        df_temp = stocks_quotes_download(isin, '2012-01-01')
        all_data.append(df_temp)
        print(f"Processed {isin}: {df_temp.shape}")
    except Exception as e:
        print(f"Error processing {isin}: {e}")
        
combined_df = pd.concat(all_data, ignore_index=True)
print(combined_df.shape)

In [ ]:
combined_df = pd.concat(all_data, ignore_index=True)
print(combined_df.shape)

## Downloading all data on each date for stocks and bonds

In [ ]:
import requests
import pandas as pd

def trading_history(date='2025-04-29'):
    url = "https://iss.moex.com/iss/history/engines/stock/markets/shares/securities.json"
    start = 0
    rows, cols = [], None

    while True:
        params = {
            'marketprice_board':1,
            'date':date,
            "start":start,
        }
        resp = requests.get(url, params=params, timeout = 10).json()["history"]
        data = resp["data"]
        if not data:
            break
        if cols is None:
            cols = resp["columns"]
        rows += data
        start += len(data)

    return pd.DataFrame(rows, columns=cols)

In [ ]:
def safe_trading_history(date, retries=50):
    for attempt in range(retries):
        try:
            return trading_history(date)
        except Exception as e:
            print(f"Error processing {date}: {e} (attempt {attempt+1} of {retries})")
            # Optionally, wait a bit before reattempting
            time.sleep(1)
    # If all attempts fail, return an empty DataFrame
    return pd.DataFrame()

from datetime import datetime, timedelta
import pandas as pd
import concurrent.futures
from tqdm import tqdm
import time

# Generate a list of date strings from 2012-01-01 to today
dates = pd.date_range("2012-01-01", datetime.today()).strftime("%Y-%m-%d").tolist()

# Use multithreading (with safe reattempts) to fetch data with a progress bar.
with concurrent.futures.ThreadPoolExecutor(max_workers=40) as executor:
    dfs = list(tqdm(executor.map(safe_trading_history, dates), total=len(dates)))
df_all = pd.concat(dfs, ignore_index=True)

In [ ]:
df_all.to_csv(r'../../Data/quotes_data/Shares.csv', index=False, encoding = 'utf-16', sep = '\t')

In [ ]:
# df_all.to_excel(r'../../Data/quotes_data/Stocks.xlsx', index=False)

In [ ]:
import requests
import pandas as pd

def instruments_info():
    url = "https://iss.moex.com/iss/securitygroups/stock_shares/collections/stock_shares_all/securities.json"
    start = 0
    rows, cols = [], None

    while True:
        params = {
            "start":start
        }
        resp = requests.get(url, params=params, timeout = 10).json()
        data = resp["securities"]["data"]
        if not data:
            break
        if cols is None:
            cols = resp["securities"]["columns"]
        rows += data
        start += len(data)
        print(start)

    return pd.DataFrame(rows, columns=cols)

In [ ]:
shares_info = instruments_info()

In [ ]:
# df_all2.to_csv(r'../../Data/quotes_data/Shares.csv', index=False, encoding = 'utf-16', sep = '\t')

In [ ]:
# df_all2 = pd.merge(df_all, shares_info[['SECID','ISIN']], how= 'left', on='SECID')